# Notebook 11 — EDA and Final Predictor Selection (Version 2)

This notebook repeats the exploratory audit using only Version 2 development
reference rows (2001Q2–2018Q4) and predictors measured at $t-1$. It is an
outcome-blind feature-governance notebook: it does not load the target, the
2019 purge rows, either version's final-test rows or final-test performance.

The default specification is the locked Version 1 manifest after the
mechanical `_lag1` rename: 19 bank predictors (including
`insufficient_history_5q_lag1`) and four existing lag-one macro predictors.
No predictor may be added. A locked predictor may be removed only if the
lagged-panel audit establishes an availability, validity or redundancy failure;
any such change must be recorded as a Version 1–Version 2 comparison
confounder. Correlation and VIF thresholds are review flags, not automatic
deletion rules.

Predictor-selection data overlap the later crisis analysis. Notebook 17 must
therefore treat this feature set as pre-locked and must not independently
reselect predictors.


In [ ]:
# Shared contract, path resolution and precommitted audit rules

from pathlib import Path
import hashlib
import importlib.util
import json
import os

import numpy as np
import pandas as pd
from IPython.display import display


COLAB_MYDRIVE = Path("/content/drive/MyDrive")
try:
    from google.colab import drive
except ImportError:
    drive = None

if COLAB_MYDRIVE.is_dir():
    print("Google Drive is already mounted at /content/drive.")
elif drive is not None:
    try:
        drive.mount("/content/drive", force_remount=False)
    except (ValueError, OSError) as exc:
        if COLAB_MYDRIVE.is_dir():
            print("Google Drive became available after the mount warning; continuing.")
        else:
            raise RuntimeError(
                "Google Drive could not be mounted. In Colab, open the Files pane, "
                "click Mount Drive, complete authorisation, and rerun this cell."
            ) from exc

DEFAULT_PROJECT_ROOT = Path(
    os.environ.get("DISSERTATION_PROJECT_ROOT", "/content/drive/MyDrive/Data Final")
).expanduser()
explicit_config = os.environ.get("V2_CONFIG_PATH")
CONFIG_PATH = (
    Path(explicit_config).expanduser()
    if explicit_config
    else DEFAULT_PROJECT_ROOT / "Version 2 Main Analysis" / "config.py"
)
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Shared Version 2 config not found: {CONFIG_PATH}")

config_spec = importlib.util.spec_from_file_location("v2_config", CONFIG_PATH)
config = importlib.util.module_from_spec(config_spec)
config_spec.loader.exec_module(config)

REQUIRED_CONFIG_FIELDS = {
    "ANALYSIS_VERSION", "VERSION_2_MAIN_TIMING_POLICY", "V2_OUTPUT_ROOT",
    "VALIDATION_FOLDS", "FINAL_TEST_DESIGN", "TARGET_HORIZON_DAYS",
}
missing_config_fields = sorted(REQUIRED_CONFIG_FIELDS.difference(dir(config)))
if missing_config_fields:
    raise AttributeError("Shared config missing: " + ", ".join(missing_config_fields))

ANALYSIS_VERSION = config.ANALYSIS_VERSION
TIMING_POLICY = dict(config.VERSION_2_MAIN_TIMING_POLICY)
VALIDATION_FOLDS = tuple(dict(item) for item in config.VALIDATION_FOLDS)
FINAL_TEST_DESIGN = dict(config.FINAL_TEST_DESIGN)
TARGET_HORIZON_DAYS = int(config.TARGET_HORIZON_DAYS)
V2_OUTPUT_ROOT = Path(config.V2_OUTPUT_ROOT).expanduser()
PROJECT_ROOT = Path(getattr(config, "PROJECT_ROOT", CONFIG_PATH.parent.parent)).expanduser()
V2_ROOT = Path(getattr(config, "V2_ROOT", CONFIG_PATH.parent)).expanduser()


def quarter_end(label):
    return pd.Period(str(label), freq="Q").end_time.normalize()


def deduplicate_paths(paths):
    result = []
    for item in paths:
        item = Path(item).expanduser()
        if item not in result:
            result.append(item)
    return result


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def passed_values(series):
    # Parse persisted gates without treating the string 'False' as true.
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "pass"})


candidate_roots = deduplicate_paths([
    V2_OUTPUT_ROOT / ANALYSIS_VERSION,
    V2_OUTPUT_ROOT,
    V2_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    V2_ROOT / "Notebook Outputs",
    PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    PROJECT_ROOT / "Notebook Outputs",
    DEFAULT_PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    DEFAULT_PROJECT_ROOT / "Notebook Outputs",
])

bundle_paths = {
    "panel": Path("09_Final_Modelling_Panel") / "v2_final_modelling_panel_2001Q2_2025Q2.parquet",
    "candidate_manifest": Path("09_Final_Modelling_Panel") / "v2_notebook09_feature_manifest.csv",
    "exclusions": Path("09_Final_Modelling_Panel") / "v2_exclusion_summary.csv",
    "membership": Path("10_Chronological_Validation_Design") / "v2_chronological_row_membership.parquet",
    "design": Path("10_Chronological_Validation_Design") / "v2_chronological_design.json",
    "nb10_gate": Path("10_Chronological_Validation_Design") / "cell_07_final_validation_gate.csv",
}


def bundle_complete(root):
    return all((root / rel).is_file() for rel in bundle_paths.values())


complete_roots = [root for root in candidate_roots if bundle_complete(root)]
if not complete_roots:
    for search_root in deduplicate_paths([
        V2_OUTPUT_ROOT, V2_ROOT / "Notebook Outputs",
        PROJECT_ROOT / "Notebook Outputs", DEFAULT_PROJECT_ROOT / "Notebook Outputs",
    ]):
        if not search_root.is_dir():
            continue
        for panel_candidate in search_root.rglob("v2_final_modelling_panel_2001Q2_2025Q2.parquet"):
            inferred_root = panel_candidate.parent.parent
            if bundle_complete(inferred_root) and inferred_root not in complete_roots:
                complete_roots.append(inferred_root)

if not complete_roots:
    raise FileNotFoundError(
        "No complete Version 2 Notebook 09/10 handoff bundle was found. Checked:\n"
        + "\n".join(str(path) for path in candidate_roots)
    )

VERSION_OUTPUT_ROOT = complete_roots[0]
NB09_OUTPUT_DIR = VERSION_OUTPUT_ROOT / "09_Final_Modelling_Panel"
NB10_OUTPUT_DIR = VERSION_OUTPUT_ROOT / "10_Chronological_Validation_Design"
NB11_OUTPUT_DIR = VERSION_OUTPUT_ROOT / "11_EDA_and_Final_Predictor_Selection"
NB11_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PANEL_PATH = VERSION_OUTPUT_ROOT / bundle_paths["panel"]
NB09_CANDIDATE_MANIFEST_PATH = VERSION_OUTPUT_ROOT / bundle_paths["candidate_manifest"]
NB09_EXCLUSION_SUMMARY_PATH = VERSION_OUTPUT_ROOT / bundle_paths["exclusions"]
NB10_MEMBERSHIP_PATH = VERSION_OUTPUT_ROOT / bundle_paths["membership"]
NB10_DESIGN_PATH = VERSION_OUTPUT_ROOT / bundle_paths["design"]
NB10_GATE_PATH = VERSION_OUTPUT_ROOT / bundle_paths["nb10_gate"]

locked_v1_relative = (
    Path("05_Quarterly_Flows_and_Candidate_Predictors")
    / "locked_v1_bank_feature_manifest_v2.csv"
)
manifest_roots = deduplicate_paths([
    V2_OUTPUT_ROOT, V2_OUTPUT_ROOT / ANALYSIS_VERSION,
    V2_ROOT / "Notebook Outputs", V2_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    PROJECT_ROOT / "Notebook Outputs", PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    DEFAULT_PROJECT_ROOT / "Notebook Outputs", DEFAULT_PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
])
locked_manifest_candidates = [root / locked_v1_relative for root in manifest_roots]
LOCKED_V1_BANK_MANIFEST_PATH = next(
    (path for path in locked_manifest_candidates if path.is_file()), None
)
if LOCKED_V1_BANK_MANIFEST_PATH is None:
    raise FileNotFoundError(
        "Authoritative Notebook 05 locked Version 1 manifest not found. Checked:\n"
        + "\n".join(str(path) for path in locked_manifest_candidates)
    )

DEVELOPMENT_START = quarter_end("2001Q2")
DEVELOPMENT_END = quarter_end(FINAL_TEST_DESIGN["train_end"])
PURGE_START = quarter_end(FINAL_TEST_DESIGN["purge_start"])
PURGE_END = quarter_end(FINAL_TEST_DESIGN["purge_end"])
TEST_START = quarter_end(FINAL_TEST_DESIGN["test_start"])
TEST_END = quarter_end(FINAL_TEST_DESIGN["test_end"])

# Locked, outcome-blind review thresholds inherited from Version 1 EDA.
MAX_MISSING_RATE = 0.40
MIN_NON_MISSING = 100
MIN_UNIQUE_VALUES = 2
CORRELATION_REVIEW_THRESHOLD = 0.90
VIF_REVIEW_THRESHOLD = 10.0
OUTLIER_IQR_MULTIPLIER = 3.0

input_panel_sha256_before = sha256_file(FINAL_PANEL_PATH)
nb10_gate = pd.read_csv(NB10_GATE_PATH)
setup_checks = {
    "Version 2 lagged-bank design": ANALYSIS_VERSION == "v2_bank_lag1_main",
    "Exact t-1 bank timing": TIMING_POLICY.get("bank_predictor_source") == "exact_t_minus_1",
    "Lag-one macro timing": TIMING_POLICY.get("macro_predictor_source") == "t_minus_1",
    "Target horizon remains 365 days": TARGET_HORIZON_DAYS == 365,
    "Development boundary is 2001Q2–2018Q4":
        DEVELOPMENT_START == pd.Timestamp("2001-06-30")
        and DEVELOPMENT_END == pd.Timestamp("2018-12-31"),
    "Notebook 10 final gate is all PASS":
        len(nb10_gate) > 0 and passed_values(nb10_gate["passed"]).all(),
    "Authoritative Notebook 05 manifest exists": LOCKED_V1_BANK_MANIFEST_PATH.is_file(),
    "Complete Notebook 09/10 bundle resolved": bundle_complete(VERSION_OUTPUT_ROOT),
}
cell_01_gate = pd.DataFrame({
    "check": setup_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in setup_checks.values()],
    "passed": setup_checks.values(),
})
cell_01_gate.to_csv(NB11_OUTPUT_DIR / "cell_01_setup_gate.csv", index=False)
print("Notebook 11 Cell 1 setup gate:")
display(cell_01_gate)
if not all(setup_checks.values()):
    raise AssertionError("Cell 1 blocked: " + "; ".join(k for k, v in setup_checks.items() if not v))
print("Resolved handoff root:", VERSION_OUTPUT_ROOT)
print("Authoritative V1 manifest:", LOCKED_V1_BANK_MANIFEST_PATH)
print("Notebook 11 output:", NB11_OUTPUT_DIR)


In [ ]:
# Lock names, load only development rows, and gate scope before EDA

locked_v1_manifest = pd.read_csv(LOCKED_V1_BANK_MANIFEST_PATH).sort_values("feature_order")
required_manifest_columns = {
    "feature_order", "feature", "version_1_status", "notebook_09_name"
}
if not required_manifest_columns.issubset(locked_v1_manifest.columns):
    raise KeyError(
        "Notebook 05 manifest is missing: "
        + ", ".join(sorted(required_manifest_columns.difference(locked_v1_manifest.columns)))
    )

locked_status = locked_v1_manifest["version_1_status"].astype(str).str.lower()
locked_mask = locked_status.str.contains("lock|retain|select", regex=True, na=False)
if int(locked_mask.sum()) == 19:
    locked_v1_manifest = locked_v1_manifest.loc[locked_mask].copy()
elif len(locked_v1_manifest) == 19:
    # The Notebook 05 artifact itself is the locked manifest; tolerate a purely
    # descriptive status vocabulary while still enforcing its exact row count.
    locked_v1_manifest = locked_v1_manifest.copy()
else:
    raise AssertionError(
        "Could not identify exactly 19 locked Version 1 bank predictors from "
        "the authoritative Notebook 05 manifest."
    )
LOCKED_V1_BANK_NAMES = locked_v1_manifest["feature"].astype(str).tolist()
BANK_ONLY_FEATURES = locked_v1_manifest["notebook_09_name"].astype(str).tolist()
MACRO_FEATURES = [
    "real_gdp_growth_lag1",
    "unemployment_rate_lag1",
    "effective_federal_funds_rate_lag1",
    "house_price_growth_lag1",
]
BANK_PLUS_MACRO_FEATURES = BANK_ONLY_FEATURES + MACRO_FEATURES

candidate_manifest = pd.read_csv(NB09_CANDIDATE_MANIFEST_PATH)
candidate_name_column = next(
    (name for name in ["v2_notebook09_feature", "feature", "notebook_09_name"]
     if name in candidate_manifest.columns), None
)
if candidate_name_column is None:
    raise KeyError("Notebook 09 candidate manifest has no recognised feature-name column.")
candidate_names = candidate_manifest[candidate_name_column].dropna().astype(str).tolist()

ID_COLUMNS = ["cert", "idrssd", "report_date"]
requested_columns = ID_COLUMNS + BANK_PLUS_MACRO_FEATURES

# The parquet predicate and immediate date filter ensure no purge/test row is
# admitted to the working frame. The target column is intentionally never read.
working = pd.read_parquet(
    FINAL_PANEL_PATH,
    columns=requested_columns,
    filters=[("report_date", ">=", DEVELOPMENT_START), ("report_date", "<=", DEVELOPMENT_END)],
)
working["report_date"] = pd.to_datetime(working["report_date"]).dt.normalize()
working = working.loc[
    working["report_date"].between(DEVELOPMENT_START, DEVELOPMENT_END, inclusive="both")
].copy()

membership_columns = [
    "panel_row_id", "cert", "idrssd", "report_date", "final_role",
    "fold1_role", "fold2_role", "fold3_role",
]
membership = pd.read_parquet(
    NB10_MEMBERSHIP_PATH,
    columns=membership_columns,
    filters=[("report_date", ">=", DEVELOPMENT_START), ("report_date", "<=", DEVELOPMENT_END)],
)
membership["report_date"] = pd.to_datetime(membership["report_date"]).dt.normalize()
membership = membership.loc[
    membership["report_date"].between(DEVELOPMENT_START, DEVELOPMENT_END, inclusive="both")
].copy()

key_columns = ["cert", "idrssd", "report_date"]
working = working.merge(membership, on=key_columns, how="left", validate="one_to_one", indicator=True)

date_quarters = working["report_date"].dt.to_period("Q")
expected_quarters = pd.period_range("2001Q2", "2018Q4", freq="Q")
scope_values = {
    "rows": int(len(working)),
    "distinct_banks": int(working["cert"].nunique()),
    "minimum_reference_date": working["report_date"].min(),
    "maximum_reference_date": working["report_date"].max(),
    "distinct_reference_quarters": int(date_quarters.nunique()),
    "rows_from_2019_purge": int(working["report_date"].between(PURGE_START, PURGE_END).sum()),
    "rows_from_final_test": int(working["report_date"].between(TEST_START, TEST_END).sum()),
    "target_column_loaded": False,
}
scope_audit = pd.DataFrame({"measure": scope_values.keys(), "value": scope_values.values()})
scope_audit.to_csv(NB11_OUTPUT_DIR / "v2_development_scope_audit.csv", index=False)

scope_checks = {
    "Authoritative locked V1 bank manifest contains exactly 19 predictors": len(BANK_ONLY_FEATURES) == 19,
    "insufficient_history_5q is among the locked V1 nineteen":
        "insufficient_history_5q" in LOCKED_V1_BANK_NAMES
        and "insufficient_history_5q_lag1" in BANK_ONLY_FEATURES,
    "Bank-plus-macro specification contains exactly 23 predictors": len(BANK_PLUS_MACRO_FEATURES) == 23,
    "Locked names are unique": len(BANK_PLUS_MACRO_FEATURES) == len(set(BANK_PLUS_MACRO_FEATURES)),
    "All locked V2 names exist in the Notebook 09 candidate manifest":
        set(BANK_PLUS_MACRO_FEATURES).issubset(candidate_names),
    "Candidate-only loans_to_deposits_lag1 is not added":
        "loans_to_deposits_lag1" in candidate_names
        and "loans_to_deposits_lag1" not in BANK_PLUS_MACRO_FEATURES,
    "Working frame begins at 2001Q2": working["report_date"].min() == DEVELOPMENT_START,
    "Working frame ends at 2018Q4": working["report_date"].max() == DEVELOPMENT_END,
    "All 71 development reference quarters are present":
        date_quarters.nunique() == len(expected_quarters)
        and set(date_quarters.unique()) == set(expected_quarters),
    "No 2019 purge reference row entered the working frame": scope_values["rows_from_2019_purge"] == 0,
    "No final-test reference row entered the working frame": scope_values["rows_from_final_test"] == 0,
    "The failure target was not loaded": "failed_1yr" not in working.columns,
    "No duplicate CERT/IDRSSD-quarter key": not working.duplicated(key_columns).any(),
    "Every row reconciles to Notebook 10 membership": working["_merge"].eq("both").all(),
    "Every development row is final-design training": working["final_role"].eq("training").all(),
}
cell_02_gate = pd.DataFrame({
    "check": scope_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in scope_checks.values()],
    "passed": scope_checks.values(),
})
cell_02_gate.to_csv(NB11_OUTPUT_DIR / "cell_02_development_scope_gate.csv", index=False)
if not all(scope_checks.values()):
    raise AssertionError("Cell 2 blocked before EDA: " + "; ".join(k for k, v in scope_checks.items() if not v))

# This is the first display of data-dependent Notebook 11 results. Scope has
# already been saved and asserted above.
print("Notebook 11 Cell 2 development-only gate (passed before all EDA):")
display(cell_02_gate)
print("Development-scope audit:")
display(scope_audit)
working = working.drop(columns="_merge")


In [ ]:
# Availability, missingness, reporting changes and exact-predecessor exclusions

availability_rows = []
for feature in BANK_PLUS_MACRO_FEATURES:
    values = pd.to_numeric(working[feature], errors="coerce")
    finite = values.replace([np.inf, -np.inf], np.nan)
    observed = finite.notna()
    availability_rows.append({
        "feature": feature,
        "feature_group": "bank" if feature in BANK_ONLY_FEATURES else "macro",
        "observations": len(values),
        "non_missing": int(observed.sum()),
        "missing": int((~observed).sum()),
        "missing_rate": float((~observed).mean()),
        "unique_non_missing": int(finite.nunique(dropna=True)),
        "positive_infinity_count": int(np.isposinf(values).sum()),
        "negative_infinity_count": int(np.isneginf(values).sum()),
        "first_non_missing_reference_date": working.loc[observed, "report_date"].min(),
        "last_non_missing_reference_date": working.loc[observed, "report_date"].max(),
        "passes_missingness_rule": float((~observed).mean()) <= MAX_MISSING_RATE,
        "passes_minimum_observations": int(observed.sum()) >= MIN_NON_MISSING,
        "passes_information_rule": int(finite.nunique(dropna=True)) >= MIN_UNIQUE_VALUES,
        "passes_finite_value_rule": int((np.isinf(values)).sum()) == 0,
    })
availability = pd.DataFrame(availability_rows)
availability.to_csv(NB11_OUTPUT_DIR / "v2_predictor_availability_and_missingness.csv", index=False)

quarterly_missingness = (
    working.assign(reference_quarter=working["report_date"].dt.to_period("Q").astype(str))
    .groupby("reference_quarter", sort=True)[BANK_PLUS_MACRO_FEATURES]
    .agg(lambda series: series.isna().mean())
    .reset_index()
)
quarterly_missingness.to_csv(NB11_OUTPUT_DIR / "v2_quarterly_predictor_missingness.csv", index=False)

yearly_missingness = (
    working.assign(reference_year=working["report_date"].dt.year)
    .groupby("reference_year", sort=True)[BANK_PLUS_MACRO_FEATURES]
    .agg(lambda series: series.isna().mean())
    .reset_index()
)
yearly_missingness.to_csv(NB11_OUTPUT_DIR / "v2_annual_predictor_missingness.csv", index=False)

long_q_missing = quarterly_missingness.melt(
    id_vars="reference_quarter", var_name="feature", value_name="missing_rate"
)
long_q_missing["previous_quarter_missing_rate"] = long_q_missing.groupby("feature")["missing_rate"].shift()
long_q_missing["absolute_quarter_change"] = (
    long_q_missing["missing_rate"] - long_q_missing["previous_quarter_missing_rate"]
).abs()
reporting_change_review = long_q_missing.loc[
    long_q_missing["absolute_quarter_change"].ge(0.25)
].sort_values(["absolute_quarter_change", "feature"], ascending=[False, True])
reporting_change_review.to_csv(NB11_OUTPUT_DIR / "v2_structural_reporting_change_review.csv", index=False)

history_feature = "insufficient_history_5q_lag1"
history_audit_features = [
    feature for feature in BANK_PLUS_MACRO_FEATURES if feature != history_feature
]
missing_by_history = (
    working.groupby(history_feature, dropna=False)[history_audit_features]
    .agg(lambda series: series.isna().mean())
    .reset_index()
)
missing_by_history.to_csv(NB11_OUTPUT_DIR / "v2_missingness_by_history_indicator.csv", index=False)

exclusions = pd.read_csv(NB09_EXCLUSION_SUMMARY_PATH)
exact_exclusion_mask = pd.Series(False, index=exclusions.index)
for column in exclusions.columns:
    exact_exclusion_mask |= exclusions[column].astype(str).str.contains(
        "exact|predecessor|tminus1|t_minus_1", case=False, regex=True, na=False
    )
exact_exclusions = exclusions.loc[exact_exclusion_mask].copy()
exact_exclusions.to_csv(NB11_OUTPUT_DIR / "v2_exact_predecessor_exclusion_audit.csv", index=False)

availability_checks = {
    "All 23 locked predictors were audited": set(availability["feature"]) == set(BANK_PLUS_MACRO_FEATURES),
    "No predictor contains infinity": availability["passes_finite_value_rule"].all(),
    "Quarterly missingness covers all 71 development quarters": len(quarterly_missingness) == 71,
    "Notebook 09 exact-predecessor exclusions are represented": len(exact_exclusions) > 0,
}
cell_03_gate = pd.DataFrame({
    "check": availability_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in availability_checks.values()],
    "passed": availability_checks.values(),
})
cell_03_gate.to_csv(NB11_OUTPUT_DIR / "cell_03_availability_gate.csv", index=False)
print("Predictor availability and missingness:")
display(availability)
print("Potential structural reporting changes (absolute quarterly missingness change ≥ 25 pp):")
display(reporting_change_review)
print("Exact-predecessor exclusions inherited from Notebook 09:")
display(exact_exclusions)
display(cell_03_gate)
if not all(availability_checks.values()):
    raise AssertionError("Cell 3 blocked: " + "; ".join(k for k, v in availability_checks.items() if not v))


In [ ]:
# Actual structural coverage and early-window imputation burden

TRAILING_FOUR_QUARTER_FEATURES = [
    "provisions_to_average_assets_lag1",
    "return_on_average_assets_lag1",
    "net_interest_margin_lag1",
    "interest_expense_to_average_assets_lag1",
    "noninterest_income_to_average_assets_lag1",
    "efficiency_ratio_lag1",
]
YEAR_ON_YEAR_GROWTH_FEATURES = [
    "annual_asset_growth_lag1",
    "annual_loan_growth_lag1",
    "annual_deposit_growth_lag1",
]
EXPECTED_FIRST_COMPLETE = {
    "trailing_four_quarter": pd.Timestamp("2002-03-31"),
    "year_on_year_growth": pd.Timestamp("2002-06-30"),
}

# Quarter-level coverage distinguishes structural support from incidental
# bank-level missingness. A family becomes structurally complete when every
# member has at least one observed value in the actual reference-quarter rows;
# the cell-coverage rate is reported alongside that support test.
quarter_coverage_rows = []
for reference_date, quarter_frame in working.groupby("report_date", sort=True):
    for family, features in {
        "trailing_four_quarter": TRAILING_FOUR_QUARTER_FEATURES,
        "year_on_year_growth": YEAR_ON_YEAR_GROWTH_FEATURES,
    }.items():
        observed_by_feature = quarter_frame[features].notna().sum()
        quarter_coverage_rows.append({
            "reference_date": reference_date,
            "reference_quarter": str(reference_date.to_period("Q")),
            "family": family,
            "quarter_rows": len(quarter_frame),
            "family_predictors": len(features),
            "predictors_with_any_observed_value": int(observed_by_feature.gt(0).sum()),
            "predictors_with_full_row_coverage": int(observed_by_feature.eq(len(quarter_frame)).sum()),
            "observed_predictor_cells": int(quarter_frame[features].notna().sum().sum()),
            "possible_predictor_cells": int(len(quarter_frame) * len(features)),
            "actual_cell_coverage_share": float(quarter_frame[features].notna().to_numpy().mean()),
            "all_family_predictors_structurally_supported": bool(observed_by_feature.gt(0).all()),
        })
quarter_family_coverage = pd.DataFrame(quarter_coverage_rows)
first_structurally_complete = (
    quarter_family_coverage.loc[quarter_family_coverage["all_family_predictors_structurally_supported"]]
    .groupby("family")["reference_date"].min()
)
quarter_family_coverage.to_csv(NB11_OUTPUT_DIR / "v2_structural_actual_quarter_coverage.csv", index=False)

structural_rows = []
for family, features in {
    "trailing_four_quarter": TRAILING_FOUR_QUARTER_FEATURES,
    "year_on_year_growth": YEAR_ON_YEAR_GROWTH_FEATURES,
}.items():
    expected_date = EXPECTED_FIRST_COMPLETE[family]
    for feature in features:
        values = pd.to_numeric(working[feature], errors="coerce")
        observed_dates = working.loc[values.notna(), "report_date"]
        actual_first = observed_dates.min()
        panel_start_mask = working["report_date"].eq(DEVELOPMENT_START)
        before_support_mask = working["report_date"].lt(expected_date)
        structural_rows.append({
            "family": family,
            "feature": feature,
            "expected_first_complete_reference_quarter": expected_date,
            "actual_first_non_missing_reference_quarter": actual_first,
            "actual_matches_expected": actual_first == expected_date,
            "panel_start_rows": int(panel_start_mask.sum()),
            "panel_start_missing_rows": int(values.loc[panel_start_mask].isna().sum()),
            "panel_start_structurally_incomplete_share": float(values.loc[panel_start_mask].isna().mean()),
            "pre_support_rows": int(before_support_mask.sum()),
            "pre_support_missing_rows": int(values.loc[before_support_mask].isna().sum()),
            "pre_support_missing_share": float(values.loc[before_support_mask].isna().mean()),
        })
structural_coverage = pd.DataFrame(structural_rows)
structural_coverage.to_csv(NB11_OUTPUT_DIR / "v2_structural_start_coverage.csv", index=False)

structural_family_summary = (
    structural_coverage.groupby("family", as_index=False)
    .agg(
        predictors=("feature", "size"),
        expected_first_complete_reference_quarter=("expected_first_complete_reference_quarter", "first"),
        actual_first_complete_reference_quarter=("actual_first_non_missing_reference_quarter", "max"),
        predictors_matching_expected=("actual_matches_expected", "sum"),
        predictors_structurally_incomplete_at_panel_start=("panel_start_missing_rows", lambda values: int((values > 0).sum())),
        mean_panel_start_incomplete_share=("panel_start_structurally_incomplete_share", "mean"),
        aggregate_panel_start_incomplete_share=("panel_start_missing_rows", "sum"),
        aggregate_panel_start_cells=("panel_start_rows", "sum"),
    )
)
structural_family_summary["share_of_predictors_structurally_incomplete_at_panel_start"] = (
    structural_family_summary["predictors_structurally_incomplete_at_panel_start"]
    / structural_family_summary["predictors"]
)
structural_family_summary["aggregate_panel_start_incomplete_share"] = (
    structural_family_summary["aggregate_panel_start_incomplete_share"]
    / structural_family_summary["aggregate_panel_start_cells"]
)
structural_family_summary.to_csv(NB11_OUTPUT_DIR / "v2_structural_start_family_summary.csv", index=False)

boundary_rows = []
for family, features in {
    "trailing_four_quarter": TRAILING_FOUR_QUARTER_FEATURES,
    "year_on_year_growth": YEAR_ON_YEAR_GROWTH_FEATURES,
}.items():
    expected_date = EXPECTED_FIRST_COMPLETE[family]
    actual_date = first_structurally_complete.get(family, pd.NaT)
    expected_quarter_frame = working.loc[working["report_date"].eq(expected_date), features]
    available_at_expected = [
        feature for feature in features if expected_quarter_frame[feature].notna().any()
    ]
    unavailable_at_expected = [feature for feature in features if feature not in available_at_expected]
    boundary_match = bool(pd.notna(actual_date) and actual_date == expected_date)
    if boundary_match:
        finding = "Pre-specified boundary confirmed by actual quarter coverage."
    else:
        finding = (
            "Pre-specified boundary not supported by the lagged panel. "
            f"At {expected_date.to_period('Q')}, supported predictors were "
            f"{available_at_expected}; unsupported predictors were {unavailable_at_expected}. "
            f"The complete family first appears at "
            f"{actual_date.to_period('Q') if pd.notna(actual_date) else 'never'}."
        )
    if family == "trailing_four_quarter" and not boundary_match:
        mechanism_note = (
            "The observed pattern is consistent with the five average-assets-normalised ratios "
            "requiring additional denominator history: after the t-1 timing shift they first "
            "appear at reference 2002Q2, while efficiency_ratio_lag1 is already supported in 2002Q1."
        )
    else:
        mechanism_note = "No boundary discrepancy requiring a mechanism note."
    boundary_rows.append({
        "family": family,
        "pre_specified_expected_reference_quarter": expected_date,
        "actual_first_structurally_complete_reference_quarter": actual_date,
        "expected_boundary_confirmed": boundary_match,
        "predictors_available_at_expected_boundary": " | ".join(available_at_expected),
        "predictors_unavailable_at_expected_boundary": " | ".join(unavailable_at_expected),
        "finding": finding,
        "mechanism_note": mechanism_note,
        "feature_decision": (
            "Retain locked features; the discrepancy is confined to early structural availability, "
            "is quantified in the imputation audit and does not justify an outcome-informed feature change."
        ),
        "version_1_version_2_comparison_limitation": not boundary_match,
    })
structural_boundary_reconciliation = pd.DataFrame(boundary_rows)
structural_boundary_reconciliation.to_csv(
    NB11_OUTPUT_DIR / "v2_structural_boundary_reconciliation.csv", index=False
)

# Imputation is quantified, not performed. Fold 1 training medians are used only
# to establish that a training-only median would be available downstream.
fold1_training = working.loc[working["fold1_role"].eq("training")].copy()
fold1_medians = fold1_training[BANK_PLUS_MACRO_FEATURES].median(numeric_only=True)
early_mask = working["report_date"].between(
    pd.Timestamp("2001-06-30"), pd.Timestamp("2002-03-31"), inclusive="both"
)
early_rows = []
for feature in BANK_PLUS_MACRO_FEATURES:
    missing_cells = int(working.loc[early_mask, feature].isna().sum())
    total_cells = int(early_mask.sum())
    early_rows.append({
        "feature": feature,
        "feature_group": "bank" if feature in BANK_ONLY_FEATURES else "macro",
        "early_window": "2001Q2–2002Q1",
        "early_rows": total_cells,
        "missing_cells_requiring_training_median": missing_cells,
        "training_median_imputation_share": missing_cells / total_cells if total_cells else np.nan,
        "fold1_training_median": fold1_medians.get(feature, np.nan),
        "fold1_training_median_available": pd.notna(fold1_medians.get(feature, np.nan)),
    })
early_imputation = pd.DataFrame(early_rows)
early_imputation.to_csv(NB11_OUTPUT_DIR / "v2_early_window_training_median_imputation_share.csv", index=False)

early_imputation_summary = []
for name, features in {
    "bank_only_19": BANK_ONLY_FEATURES,
    "bank_plus_macro_23": BANK_PLUS_MACRO_FEATURES,
    "trailing_four_quarter": TRAILING_FOUR_QUARTER_FEATURES,
    "year_on_year_growth": YEAR_ON_YEAR_GROWTH_FEATURES,
}.items():
    part = early_imputation.loc[early_imputation["feature"].isin(features)]
    early_imputation_summary.append({
        "predictor_set": name,
        "predictors": len(features),
        "early_rows": int(early_mask.sum()),
        "predictor_cells": int(part["early_rows"].sum()),
        "imputed_cells_if_training_median_used": int(part["missing_cells_requiring_training_median"].sum()),
        "training_median_imputation_share": float(
            part["missing_cells_requiring_training_median"].sum() / part["early_rows"].sum()
        ),
    })
early_imputation_summary = pd.DataFrame(early_imputation_summary)
early_imputation_summary["limitation"] = (
    "The 2001Q2–2002Q1 development rows carry disproportionate structural missingness; "
    "downstream fold-specific training-median imputation must be fitted on training rows only."
)
early_imputation_summary.to_csv(NB11_OUTPUT_DIR / "v2_early_window_imputation_limitation.csv", index=False)

coverage_checks = {
    "Every structural feature has an observed first-support quarter":
        structural_coverage["actual_first_non_missing_reference_quarter"].notna().all(),
    "Actual family boundaries reconcile to the latest member first-support quarter":
        all(
            first_structurally_complete.get(row.family)
            == row.actual_first_complete_reference_quarter
            for row in structural_family_summary.itertuples()
        ),
    "Any pre-specified boundary discrepancy is preserved and explicitly disclosed":
        len(structural_boundary_reconciliation) == 2
        and structural_boundary_reconciliation["finding"].notna().all()
        and structural_boundary_reconciliation["feature_decision"].str.contains("Retain locked features").all(),
    "Actual quarter coverage shows year-on-year-growth structural completeness from 2002Q2":
        first_structurally_complete.get("year_on_year_growth") == pd.Timestamp("2002-06-30")
        and
        structural_coverage.loc[
            structural_coverage["family"].eq("year_on_year_growth"), "actual_matches_expected"
        ].all(),
    "Every locked predictor has an available Fold 1 training median":
        early_imputation["fold1_training_median_available"].all(),
    "Early-window imputation burden is quantified for all 23 predictors": len(early_imputation) == 23,
}
cell_04_gate = pd.DataFrame({
    "check": coverage_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in coverage_checks.values()],
    "passed": coverage_checks.values(),
})
cell_04_gate.to_csv(NB11_OUTPUT_DIR / "cell_04_structural_coverage_gate.csv", index=False)
print("Actual structural support by feature:")
display(structural_coverage)
print("Actual quarter coverage around the structural boundary:")
display(quarter_family_coverage.loc[quarter_family_coverage["reference_date"].le(pd.Timestamp("2002-09-30"))])
print("Structural-start summary:")
display(structural_family_summary)
print("Pre-specified versus actual structural-boundary reconciliation:")
display(structural_boundary_reconciliation)
print("Training-median imputation burden in 2001Q2–2002Q1 (quantification only):")
display(early_imputation_summary)
display(cell_04_gate)
if not all(coverage_checks.values()):
    raise AssertionError("Cell 4 blocked: " + "; ".join(k for k, v in coverage_checks.items() if not v))


In [ ]:
# Development-only distributions and outlier review

distribution_rows = []
outlier_rows = []
for feature in BANK_PLUS_MACRO_FEATURES:
    values = pd.to_numeric(working[feature], errors="coerce").replace([np.inf, -np.inf], np.nan)
    observed = values.dropna()
    quantiles = observed.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    distribution_rows.append({
        "feature": feature,
        "count": int(observed.size),
        "missing": int(values.isna().sum()),
        "mean": observed.mean(),
        "std": observed.std(),
        "minimum": observed.min(),
        "p01": quantiles.loc[0.01],
        "p05": quantiles.loc[0.05],
        "p25": quantiles.loc[0.25],
        "median": quantiles.loc[0.50],
        "p75": quantiles.loc[0.75],
        "p95": quantiles.loc[0.95],
        "p99": quantiles.loc[0.99],
        "maximum": observed.max(),
        "skewness": observed.skew(),
        "excess_kurtosis": observed.kurt(),
        "zero_share": float(observed.eq(0).mean()),
        "negative_share": float(observed.lt(0).mean()),
    })
    q1, q3 = quantiles.loc[0.25], quantiles.loc[0.75]
    iqr = q3 - q1
    lower = q1 - OUTLIER_IQR_MULTIPLIER * iqr
    upper = q3 + OUTLIER_IQR_MULTIPLIER * iqr
    outlier = observed.lt(lower) | observed.gt(upper)
    outlier_rows.append({
        "feature": feature,
        "iqr_multiplier": OUTLIER_IQR_MULTIPLIER,
        "lower_review_bound": lower,
        "upper_review_bound": upper,
        "outlier_count": int(outlier.sum()),
        "outlier_share_of_observed": float(outlier.mean()),
        "decision_rule": "Review only; no winsorisation or deletion is performed in Notebook 11.",
    })

distribution_summary = pd.DataFrame(distribution_rows)
outlier_review = pd.DataFrame(outlier_rows)
distribution_summary.to_csv(NB11_OUTPUT_DIR / "v2_development_distribution_summary.csv", index=False)
outlier_review.to_csv(NB11_OUTPUT_DIR / "v2_development_outlier_review.csv", index=False)

cell_05_gate = pd.DataFrame({
    "check": [
        "Distribution audit contains all 23 locked predictors",
        "Outlier audit contains all 23 locked predictors",
        "No working-frame value was modified",
    ],
    "result": ["PASS", "PASS", "PASS"],
    "passed": [
        len(distribution_summary) == 23,
        len(outlier_review) == 23,
        True,
    ],
})
cell_05_gate["result"] = np.where(cell_05_gate["passed"], "PASS", "FAIL")
cell_05_gate.to_csv(NB11_OUTPUT_DIR / "cell_05_distribution_gate.csv", index=False)
print("Development-period distribution summary:")
display(distribution_summary)
print("Outlier review (audit flags only):")
display(outlier_review)
display(cell_05_gate)
if not cell_05_gate["passed"].all():
    raise AssertionError("Cell 5 blocked.")


In [ ]:
# Spearman correlation and VIF review (EDA-only median copy)

spearman = working[BANK_PLUS_MACRO_FEATURES].corr(method="spearman", min_periods=MIN_NON_MISSING)
spearman.to_csv(NB11_OUTPUT_DIR / "v2_spearman_correlation_matrix.csv")

correlation_pairs = []
for left_index, left in enumerate(BANK_PLUS_MACRO_FEATURES):
    for right in BANK_PLUS_MACRO_FEATURES[left_index + 1:]:
        value = spearman.loc[left, right]
        correlation_pairs.append({
            "feature_1": left,
            "feature_2": right,
            "spearman_correlation": value,
            "absolute_correlation": abs(value) if pd.notna(value) else np.nan,
            "review_flag": bool(pd.notna(value) and abs(value) >= CORRELATION_REVIEW_THRESHOLD),
            "decision_rule": "Review distinct economic content; high correlation is not automatic deletion.",
        })
correlation_pairs = pd.DataFrame(correlation_pairs).sort_values(
    "absolute_correlation", ascending=False, na_position="last"
)
correlation_pairs.to_csv(NB11_OUTPUT_DIR / "v2_spearman_correlation_pairs.csv", index=False)

# This complete-case copy exists only for collinearity diagnostics. Its medians
# must never be reused by modelling notebooks, which fit preprocessing inside
# each training fold.
eda_medians = working[BANK_PLUS_MACRO_FEATURES].median(numeric_only=True)
eda_complete = working[BANK_PLUS_MACRO_FEATURES].fillna(eda_medians)
eda_median_table = pd.DataFrame({
    "feature": BANK_PLUS_MACRO_FEATURES,
    "development_median_for_eda_only": [eda_medians[name] for name in BANK_PLUS_MACRO_FEATURES],
    "permitted_downstream_use": "None — downstream preprocessing must refit inside each training fold.",
})
eda_median_table.to_csv(NB11_OUTPUT_DIR / "v2_eda_only_medians_not_for_modelling.csv", index=False)


def calculate_vif(frame, features, set_name):
    corr = frame[features].corr(method="pearson").to_numpy(dtype=float)
    rows = []
    for index, feature in enumerate(features):
        keep = [item for item in range(len(features)) if item != index]
        predictor_corr = corr[np.ix_(keep, keep)]
        cross_corr = corr[index, keep]
        r_squared = float(cross_corr @ np.linalg.pinv(predictor_corr) @ cross_corr.T)
        r_squared = min(max(r_squared, 0.0), 1.0)
        vif = np.inf if r_squared >= 1.0 - 1e-12 else 1.0 / (1.0 - r_squared)
        rows.append({
            "predictor_set": set_name,
            "feature": feature,
            "r_squared_against_other_predictors": r_squared,
            "vif": vif,
            "review_flag": bool(vif >= VIF_REVIEW_THRESHOLD),
            "decision_rule": "Review only; retain when economic content is distinct and validity remains satisfied.",
        })
    return pd.DataFrame(rows)


vif_review = pd.concat([
    calculate_vif(eda_complete, BANK_ONLY_FEATURES, "bank_only_19"),
    calculate_vif(eda_complete, BANK_PLUS_MACRO_FEATURES, "bank_plus_macro_23"),
], ignore_index=True)
vif_review.to_csv(NB11_OUTPUT_DIR / "v2_vif_review.csv", index=False)

cell_06_gate = pd.DataFrame({
    "check": [
        "Correlation matrix is 23 by 23",
        "All unique correlation pairs are present",
        "VIF covers 19 bank-only and 23 bank-plus-macro predictors",
        "EDA-only medians are explicitly prohibited from downstream reuse",
    ],
    "passed": [
        spearman.shape == (23, 23),
        len(correlation_pairs) == 23 * 22 // 2,
        (vif_review["predictor_set"].eq("bank_only_19").sum() == 19)
        and (vif_review["predictor_set"].eq("bank_plus_macro_23").sum() == 23),
        eda_median_table["permitted_downstream_use"].str.startswith("None").all(),
    ],
})
cell_06_gate.insert(1, "result", np.where(cell_06_gate["passed"], "PASS", "FAIL"))
cell_06_gate.to_csv(NB11_OUTPUT_DIR / "cell_06_redundancy_gate.csv", index=False)
print("Highest absolute Spearman correlations:")
display(correlation_pairs.head(25))
print("VIF review:")
display(vif_review.sort_values(["predictor_set", "vif"], ascending=[True, False]))
display(cell_06_gate)
if not cell_06_gate["passed"].all():
    raise AssertionError("Cell 6 blocked.")


In [ ]:
# Economic interpretation at t-1

interpretation_map = {
    "log_total_assets_lag1": ("Size", "Ambiguous", "Scale, complexity and diversification one quarter before t."),
    "equity_to_assets_lag1": ("Capital", "Lower is riskier", "Loss-absorbing capital relative to assets at t-1."),
    "noncurrent_loans_to_loans_lag1": ("Asset quality", "Higher is riskier", "Share of loans already noncurrent at t-1."),
    "allowance_to_loans_lag1": ("Asset quality", "Ambiguous", "Allowance coverage, reflecting both buffers and recognised credit risk at t-1."),
    "provisions_to_average_assets_lag1": ("Asset quality", "Higher is riskier", "Trailing-four-quarter credit-loss provisioning intensity known at t-1."),
    "other_real_estate_owned_to_assets_lag1": ("Asset quality", "Higher is riskier", "Foreclosed real-estate burden at t-1."),
    "return_on_average_assets_lag1": ("Earnings", "Lower is riskier", "Trailing-four-quarter profitability known at t-1."),
    "net_interest_margin_lag1": ("Earnings", "Usually lower is riskier", "Trailing-four-quarter core intermediation margin at t-1."),
    "interest_expense_to_average_assets_lag1": ("Earnings/Funding", "Higher is riskier", "Trailing-four-quarter funding-cost pressure at t-1."),
    "noninterest_income_to_average_assets_lag1": ("Earnings", "Ambiguous", "Trailing-four-quarter noninterest revenue contribution at t-1."),
    "efficiency_ratio_lag1": ("Management/Earnings", "Higher is riskier", "Trailing-four-quarter noninterest cost burden per revenue at t-1."),
    "loans_to_assets_lag1": ("Liquidity/Asset mix", "Higher may be riskier", "Illiquid loan concentration at t-1."),
    "deposits_to_assets_lag1": ("Funding", "Usually lower is riskier", "Deposit-funded share of assets at t-1."),
    "securities_to_assets_lag1": ("Liquidity/Asset mix", "Ambiguous", "Securities allocation and potential liquidity buffer at t-1."),
    "aoci_to_assets_lag1": ("Capital/Market risk", "More negative is riskier", "Accumulated other comprehensive income relative to assets at t-1."),
    "annual_asset_growth_lag1": ("Growth", "Rapid growth may be riskier", "Year-on-year asset expansion measured through t-1."),
    "annual_loan_growth_lag1": ("Growth", "Rapid growth may be riskier", "Year-on-year loan expansion measured through t-1."),
    "annual_deposit_growth_lag1": ("Growth/Funding", "Large decline may be riskier", "Year-on-year deposit change measured through t-1."),
    "insufficient_history_5q_lag1": ("Data history", "Indicator", "Flags insufficient bank history for five-quarter constructions at t-1."),
    "real_gdp_growth_lag1": ("Macroeconomy", "Lower is riskier", "Real economic growth in the quarter preceding t."),
    "unemployment_rate_lag1": ("Macroeconomy", "Higher is riskier", "Labour-market weakness in the quarter preceding t."),
    "effective_federal_funds_rate_lag1": ("Monetary conditions", "Ambiguous", "Short-term policy-rate environment in the quarter preceding t."),
    "house_price_growth_lag1": ("Macro/Collateral", "Lower is riskier", "Housing collateral-cycle growth in the quarter preceding t."),
}
economic_interpretation = pd.DataFrame([
    {
        "feature": feature,
        "economic_area": interpretation_map[feature][0],
        "expected_risk_direction": interpretation_map[feature][1],
        "timing": "t-1",
        "economic_interpretation": interpretation_map[feature][2],
    }
    for feature in BANK_PLUS_MACRO_FEATURES
])
economic_interpretation.to_csv(NB11_OUTPUT_DIR / "v2_predictor_economic_interpretation.csv", index=False)

cell_07_gate = pd.DataFrame({
    "check": ["Every locked predictor has a t-1 economic interpretation"],
    "passed": [
        set(economic_interpretation["feature"]) == set(BANK_PLUS_MACRO_FEATURES)
        and economic_interpretation["timing"].eq("t-1").all()
    ],
})
cell_07_gate.insert(1, "result", np.where(cell_07_gate["passed"], "PASS", "FAIL"))
cell_07_gate.to_csv(NB11_OUTPUT_DIR / "cell_07_interpretation_gate.csv", index=False)
display(economic_interpretation)
display(cell_07_gate)
if not cell_07_gate["passed"].all():
    raise AssertionError("Cell 7 blocked.")


In [ ]:
# Lock manifests and reconcile name-for-name against Version 1

failed_availability = availability.loc[
    ~(
        availability["passes_missingness_rule"]
        & availability["passes_minimum_observations"]
        & availability["passes_information_rule"]
        & availability["passes_finite_value_rule"]
    )
]
if len(failed_availability):
    display(failed_availability)
    raise AssertionError(
        "A locked feature failed availability/validity. Resolve and document the "
        "Version 1–Version 2 comparison confounder before locking Notebook 11."
    )

bank_manifest = locked_v1_manifest[["feature_order", "feature", "notebook_09_name"]].copy()
bank_manifest = bank_manifest.rename(columns={
    "feature": "version_1_feature",
    "notebook_09_name": "version_2_feature",
})
bank_manifest["feature_group"] = "bank"
bank_manifest["timing"] = "exact t-1 bank report"
bank_manifest["selection_status"] = "retained from locked Version 1 manifest"
bank_manifest["selection_data"] = "Version 2 development reference rows 2001Q2–2018Q4 only"

macro_manifest = pd.DataFrame({
    "feature_order": range(20, 24),
    "version_1_feature": MACRO_FEATURES,
    "version_2_feature": MACRO_FEATURES,
    "feature_group": "macro",
    "timing": "existing lag-one macro field",
    "selection_status": "retained from locked Version 1 manifest",
    "selection_data": "Version 2 development reference rows 2001Q2–2018Q4 only",
})
bank_plus_macro_manifest = pd.concat([bank_manifest, macro_manifest], ignore_index=True)

reconciliation = bank_plus_macro_manifest[[
    "feature_order", "version_1_feature", "version_2_feature", "feature_group"
]].copy()
reconciliation["expected_v2_feature"] = np.where(
    reconciliation["feature_group"].eq("bank"),
    reconciliation["version_1_feature"] + "_lag1",
    reconciliation["version_1_feature"],
)
reconciliation["name_match_after_required_rename"] = (
    reconciliation["version_2_feature"] == reconciliation["expected_v2_feature"]
)
reconciliation["in_locked_v1_nineteen"] = (
    reconciliation["feature_group"].eq("bank")
    & reconciliation["version_1_feature"].isin(LOCKED_V1_BANK_NAMES)
)
reconciliation["version_2_decision"] = "retain"
reconciliation["comparison_confounder"] = False

candidate_only_row = pd.DataFrame([{
    "feature_order": np.nan,
    "version_1_feature": "loans_to_deposits",
    "version_2_feature": "loans_to_deposits_lag1",
    "feature_group": "candidate-only bank ratio",
    "expected_v2_feature": "not applicable — not in locked Version 1 nineteen",
    "name_match_after_required_rename": True,
    "in_locked_v1_nineteen": False,
    "version_2_decision": "not added",
    "comparison_confounder": False,
}])
reconciliation_with_candidate_note = pd.concat([reconciliation, candidate_only_row], ignore_index=True)

change_register = pd.DataFrame([{
    "change_type": "none",
    "feature": "none",
    "evidence": (
        "All 19 locked bank and four locked macro predictors satisfied the overall Version 2 "
        "availability and validity rules. The pre-specified 2002Q1 trailing-four-quarter family "
        "boundary was not supported in the realised lagged panel and is separately disclosed."
    ),
    "comparison_confounder": False,
    "resolution": "No additions or removals; name-for-name lock after the bank _lag1 rename.",
}])

selection_policy = pd.DataFrame([
    {"control": "Outcome use", "record": "The target was not loaded; no outcome or performance informed feature decisions."},
    {"control": "Final-test use", "record": "Neither Version 1 nor Version 2 final-test rows or performance were loaded or used."},
    {"control": "Additions", "record": "No predictors were added; loans_to_deposits_lag1 remains candidate-only and excluded."},
    {"control": "Redundancy", "record": "Correlation and VIF were review diagnostics, not automatic deletion rules."},
    {"control": "Crisis-analysis overlap", "record": "Selection data overlap the later crisis analysis; Notebook 17 must not independently select predictors."},
    {"control": "Early history limitation", "record": "Structural early missingness was quantified; any downstream median imputation must be fitted inside training folds."},
    {"control": "Structural-boundary reconciliation", "record": "The pre-specified 2002Q1 trailing-four-quarter expectation is retained as an expectation and reconciled against the actual lagged-panel boundary; a mismatch is disclosed rather than overwritten."},
])

bank_manifest.to_csv(NB11_OUTPUT_DIR / "v2_bank_only_feature_manifest.csv", index=False)
bank_plus_macro_manifest.to_csv(NB11_OUTPUT_DIR / "v2_bank_plus_macro_feature_manifest.csv", index=False)
reconciliation_with_candidate_note.to_csv(NB11_OUTPUT_DIR / "v1_to_v2_feature_reconciliation.csv", index=False)
change_register.to_csv(NB11_OUTPUT_DIR / "v2_feature_change_register.csv", index=False)
selection_policy.to_csv(NB11_OUTPUT_DIR / "v2_predictor_selection_controls.csv", index=False)

locked_sets = {
    "analysis_version": ANALYSIS_VERSION,
    "development_reference_period": {"start": "2001Q2", "end": "2018Q4"},
    "bank_only_count": len(bank_manifest),
    "bank_only_features": bank_manifest["version_2_feature"].tolist(),
    "bank_plus_macro_count": len(bank_plus_macro_manifest),
    "bank_plus_macro_features": bank_plus_macro_manifest["version_2_feature"].tolist(),
    "history_indicator_was_in_locked_v1_nineteen": "insufficient_history_5q" in LOCKED_V1_BANK_NAMES,
    "additions": [],
    "removals": [],
    "final_test_performance_used": False,
    "notebook_17_independent_reselection_permitted": False,
    "structural_boundary_reconciliation": structural_boundary_reconciliation.assign(
        pre_specified_expected_reference_quarter=lambda frame: frame["pre_specified_expected_reference_quarter"].astype(str),
        actual_first_structurally_complete_reference_quarter=lambda frame: frame["actual_first_structurally_complete_reference_quarter"].astype(str),
    ).to_dict(orient="records"),
}
with open(NB11_OUTPUT_DIR / "locked_predictor_sets_v2.json", "w", encoding="utf-8") as handle:
    json.dump(locked_sets, handle, indent=2)

lock_checks = {
    "Bank-only feature set contains exactly 19 predictors": len(bank_manifest) == 19,
    "Bank-plus-macro feature set contains exactly 23 predictors": len(bank_plus_macro_manifest) == 23,
    "All names match Version 1 after the required bank _lag1 rename": reconciliation["name_match_after_required_rename"].all(),
    "insufficient_history_5q was among the locked Version 1 nineteen":
        reconciliation.loc[
            reconciliation["version_1_feature"].eq("insufficient_history_5q"),
            "in_locked_v1_nineteen",
        ].eq(True).all()
        and reconciliation["version_1_feature"].eq("insufficient_history_5q").sum() == 1,
    "No feature was added": locked_sets["additions"] == [],
    "No feature was removed": locked_sets["removals"] == [],
    "Candidate-only loans_to_deposits_lag1 is explicitly not added":
        candidate_only_row["version_2_decision"].eq("not added").all(),
    "Neither version's final-test performance informed selection": not locked_sets["final_test_performance_used"],
    "Notebook 17 independent reselection is prohibited": not locked_sets["notebook_17_independent_reselection_permitted"],
    "Structural-boundary findings are carried into the locked handoff":
        len(locked_sets["structural_boundary_reconciliation"]) == 2,
}
cell_08_gate = pd.DataFrame({
    "check": lock_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in lock_checks.values()],
    "passed": lock_checks.values(),
})
cell_08_gate.to_csv(NB11_OUTPUT_DIR / "cell_08_feature_lock_gate.csv", index=False)
print("Version 1 to Version 2 reconciliation:")
display(reconciliation_with_candidate_note)
print("Feature-lock validation gate:")
display(cell_08_gate)
if not all(lock_checks.values()):
    raise AssertionError("Cell 8 blocked: " + "; ".join(k for k, v in lock_checks.items() if not v))


In [ ]:
# Final validation, checksums and downstream handoff

input_panel_sha256_after = sha256_file(FINAL_PANEL_PATH)
gate_paths = [
    NB11_OUTPUT_DIR / "cell_01_setup_gate.csv",
    NB11_OUTPUT_DIR / "cell_02_development_scope_gate.csv",
    NB11_OUTPUT_DIR / "cell_03_availability_gate.csv",
    NB11_OUTPUT_DIR / "cell_04_structural_coverage_gate.csv",
    NB11_OUTPUT_DIR / "cell_05_distribution_gate.csv",
    NB11_OUTPUT_DIR / "cell_06_redundancy_gate.csv",
    NB11_OUTPUT_DIR / "cell_07_interpretation_gate.csv",
    NB11_OUTPUT_DIR / "cell_08_feature_lock_gate.csv",
]
earlier_gates_pass = all(passed_values(pd.read_csv(path)["passed"]).all() for path in gate_paths)

final_checks = {
    "All earlier Notebook 11 gates passed": earlier_gates_pass,
    "Notebook 09 input panel remained byte-for-byte unchanged": input_panel_sha256_before == input_panel_sha256_after,
    "Development frame contains no 2019 or final-test reference row":
        not working["report_date"].ge(PURGE_START).any(),
    "Failure target was never loaded": "failed_1yr" not in working.columns,
    "Locked bank-only manifest re-reads with exactly 19 names":
        len(pd.read_csv(NB11_OUTPUT_DIR / "v2_bank_only_feature_manifest.csv")) == 19,
    "Locked bank-plus-macro manifest re-reads with exactly 23 names":
        len(pd.read_csv(NB11_OUTPUT_DIR / "v2_bank_plus_macro_feature_manifest.csv")) == 23,
    "No addition or removal is recorded":
        pd.read_csv(NB11_OUTPUT_DIR / "v2_feature_change_register.csv")["change_type"].eq("none").all(),
    "The structural-boundary reconciliation is persisted":
        len(pd.read_csv(NB11_OUTPUT_DIR / "v2_structural_boundary_reconciliation.csv")) == 2,
}
cell_09_gate = pd.DataFrame({
    "check": final_checks.keys(),
    "result": ["PASS" if value else "FAIL" for value in final_checks.values()],
    "passed": final_checks.values(),
})
cell_09_gate.to_csv(NB11_OUTPUT_DIR / "cell_09_final_validation_gate.csv", index=False)
if not all(final_checks.values()):
    raise AssertionError("Cell 9 blocked: " + "; ".join(k for k, v in final_checks.items() if not v))

early_bank_share = early_imputation_summary.loc[
    early_imputation_summary["predictor_set"].eq("bank_only_19"),
    "training_median_imputation_share",
].iloc[0]
early_all_share = early_imputation_summary.loc[
    early_imputation_summary["predictor_set"].eq("bank_plus_macro_23"),
    "training_median_imputation_share",
].iloc[0]
final_summary = pd.DataFrame([
    ("Analysis version", ANALYSIS_VERSION),
    ("Reference-date anchor", "report_date t"),
    ("Predictor timing", "bank and macro predictors at t-1"),
    ("Development reference period", "2001Q2–2018Q4"),
    ("Development observations", len(working)),
    ("Development reference quarters", working["report_date"].dt.to_period("Q").nunique()),
    ("Rows from 2019 purge", 0),
    ("Rows from 2020Q1–2025Q2 final test", 0),
    ("Target loaded", False),
    ("Bank-only predictors", len(bank_manifest)),
    ("Bank-plus-macro predictors", len(bank_plus_macro_manifest)),
    ("insufficient_history_5q in locked V1 nineteen", True),
    ("Trailing-four-quarter pre-specified expected complete reference quarter", "2002Q1"),
    ("Trailing-four-quarter actual first complete reference quarter",
        str(first_structurally_complete.get("trailing_four_quarter").to_period("Q"))),
    ("Trailing-four-quarter expected boundary confirmed",
        bool(structural_boundary_reconciliation.loc[
            structural_boundary_reconciliation["family"].eq("trailing_four_quarter"),
            "expected_boundary_confirmed",
        ].iloc[0])),
    ("Year-on-year growth pre-specified expected complete reference quarter", "2002Q2"),
    ("Year-on-year growth actual first complete reference quarter",
        str(first_structurally_complete.get("year_on_year_growth").to_period("Q"))),
    ("Year-on-year growth expected boundary confirmed",
        bool(structural_boundary_reconciliation.loc[
            structural_boundary_reconciliation["family"].eq("year_on_year_growth"),
            "expected_boundary_confirmed",
        ].iloc[0])),
    ("2001Q2–2002Q1 bank-only training-median imputation share", early_bank_share),
    ("2001Q2–2002Q1 bank-plus-macro training-median imputation share", early_all_share),
    ("Feature additions", 0),
    ("Feature removals", 0),
    ("Notebook 17 independent predictor selection permitted", False),
    ("Input panel modified", input_panel_sha256_before != input_panel_sha256_after),
    ("Input panel SHA-256", input_panel_sha256_after),
    ("Bank-only handoff", "v2_bank_only_feature_manifest.csv"),
    ("Bank-plus-macro handoff", "v2_bank_plus_macro_feature_manifest.csv"),
], columns=["measure", "value"])
final_summary.to_csv(NB11_OUTPUT_DIR / "notebook_11_final_handoff_summary.csv", index=False)

manifest_rows = []
for path in sorted(NB11_OUTPUT_DIR.iterdir(), key=lambda item: item.name):
    if path.is_file() and path.name != "notebook_11_output_manifest.csv":
        manifest_rows.append({
            "file": path.name,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })
output_manifest = pd.DataFrame(manifest_rows)
output_manifest.to_csv(NB11_OUTPUT_DIR / "notebook_11_output_manifest.csv", index=False)

print("Notebook 11 Cell 9 final validation gate:")
display(cell_09_gate)
print("Notebook 11 final handoff summary:")
display(final_summary)
print("Output manifest:")
display(output_manifest)
print("\nCell 9 completed successfully.")
print("Notebook 11 is PASS / LOCKED.")
print("Bank-only handoff:", NB11_OUTPUT_DIR / "v2_bank_only_feature_manifest.csv")
print("Bank-plus-macro handoff:", NB11_OUTPUT_DIR / "v2_bank_plus_macro_feature_manifest.csv")
print("Notebook 12 may use these locked manifests; it must fit preprocessing inside training folds.")
